# Final Project — 分析結果瀏覽器

## 結構

| Part | 功能 | 變數 |
|---|---|---|
| **0** | 載入所有檔案 | — |
| **1** | 所有 cluster 統計總覽 + meme_quality 排行 | — |
| **2** | 單一 Cluster 深入瀏覽（留言 + 場景 + 關鍵字） | `CLUSTER_ID` |
| **3** | 留言 → 場景對應（每則留言對最近的幕） | `CLUSTER_ID` |
| **4** | 場景瀏覽器（反查：這一幕對應哪些群？） | `EPISODE`, `SCENE_NUMBER` |
| **5** | **引用 vs 改編分類** | — |
| **6** | **場景熱度排行** | — |
| **7** | **潛力台詞預測** | — |

**前置步驟（必須先跑完）**：
```
embed_lines.py  →  cluster.py  →  analyze.py  →  classify_quote_vs_remix.py  →  scene_heatmap.py  →  meme_potential.py
或：uv run final_project/run_all.py
```

---
## Part 0：載入所有檔案

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

DATA   = Path("data")
OUTPUT = Path("output")

# 輸入資料
comments  = pd.read_parquet(DATA / "comments_with_embedding.parquet")
scenes    = pd.read_parquet(DATA / "scripts_with_embedding.parquet")
lines     = pd.read_parquet(DATA / "lines_with_embedding.parquet")

# Pipeline 產出
clusters  = pd.read_parquet(DATA / "clusters.parquet")             # 每則留言 + cluster_label
analysis  = pd.read_parquet(DATA / "cluster_analysis.parquet")      # 每群統計 + scene + keywords + quality
cmt_scene = pd.read_parquet(DATA / "comment_scene_mapping.parquet") # 每則留言對應最近場景

# 三個新分析
quote_cls = pd.read_csv(OUTPUT / "quote_classification.csv")
scene_hm  = pd.read_csv(OUTPUT / "scene_heatmap.csv")
potential = pd.read_csv(OUTPUT / "potential_predictions.csv")

print(f"留言：{len(clusters)}（含 noise）")
print(f"有效 cluster：{analysis['cluster_id'].nunique()}")
print(f"Noise 數：{(clusters['cluster_label']==-1).sum()}")
print(f"場景：{len(scenes)}")
print(f"對白行：{len(lines)}")

---
## Part 1：所有 cluster 總覽 + meme_quality 排行

依 `meme_quality` 排序的完整 cluster 清單。

In [ ]:
# 顯示 cluster 排行（meme_quality 從高到低）
summary = analysis[[
    "cluster_id", "size", "likes_total", "likes_max",
    "meme_quality", "main_scene", "main_scene_sim", "keywords"
]].copy()
summary["meme_quality"]   = summary["meme_quality"].round(3)
summary["main_scene_sim"] = summary["main_scene_sim"].round(3)

pd.set_option("display.max_colwidth", 60)
pd.set_option("display.max_rows", 100)
summary

In [ ]:
# Top 15 cluster 詳細視圖
print("▌ Top 15 cluster by meme_quality\n")
for _, r in analysis.head(15).iterrows():
    print(f"C{int(r['cluster_id']):>3}  size={int(r['size']):>3}  likes={int(r['likes_total']):>6}  "
          f"quality={r['meme_quality']:.3f}  → {r['main_scene']}")
    print(f"      keywords: {r['keywords'][:70]}")
    top3 = str(r['top3_comments']).split(' ||| ')
    for i, t in enumerate(top3[:3], 1):
        print(f"      [{i}] {t[:80]}")
    print()

---
## Part 2：單一 Cluster 深入瀏覽

改 `CLUSTER_ID` 看任一群的詳細資訊（所有留言、關鍵字、場景）。

In [ ]:
CLUSTER_ID = 1     # ← 改這裡（參考 Part 1 表格的 cluster_id）

row = analysis[analysis["cluster_id"] == CLUSTER_ID]
if row.empty:
    print(f"找不到 Cluster {CLUSTER_ID}")
else:
    r = row.iloc[0]
    print(f"{'='*70}")
    print(f"Cluster {CLUSTER_ID}")
    print(f"{'='*70}")
    print(f"  size:          {r['size']} 則留言")
    print(f"  likes_total:   {r['likes_total']:,}")
    print(f"  likes_max:     {r['likes_max']:,}")
    print(f"  meme_quality:  {r['meme_quality']:.3f}")
    print(f"    - engagement:   {r['engage_n']:.3f}")
    print(f"    - keyword:      {r['kw_n']:.3f}")
    print(f"    - scene:        {r['scene_n']:.3f}")
    print(f"    - variation:    {r['var_n']:.3f}")
    print(f"  main_scene:    {r['main_scene']} (sim={r['main_scene_sim']:.3f})")
    print(f"  keywords:      {r['keywords']}")
    print()
    print(f"── 全部留言（依讚數排序）──")
    grp = clusters[clusters["cluster_label"] == CLUSTER_ID].sort_values("like_count", ascending=False)
    pd.set_option("display.max_colwidth", 120)
    display(grp[["like_count", "text_clean"]].reset_index(drop=True))

---
## Part 3：留言 → 場景對應

查看某個 cluster 的留言分別對應到哪些場景（不是 centroid，是每則留言個別判斷）。

In [ ]:
CLUSTER_ID = 1     # ← 改這裡

grp = cmt_scene[cmt_scene["cluster_label"] == CLUSTER_ID].sort_values("like_count", ascending=False)

print(f"Cluster {CLUSTER_ID} 的留言場景分布：")
print(grp["best_scene"].value_counts().to_string())

print(f"\nTop 20 高讚留言對應場景：\n")
print(f"{'讚數':>6}  {'場景':12}  {'sim':5}  留言")
print("─" * 90)
for _, r in grp.head(20).iterrows():
    print(f"{int(r['like_count']):>6}  {r['best_scene']:12}  {r['best_scene_sim']:.3f}  {str(r['text_clean'])[:55]}")

---
## Part 4：場景瀏覽器

從場景的角度反查：**這一幕吸引了哪些 cluster？留言在說什麼？**

In [ ]:
# 所有可選場景一覽
scene_overview = (
    cmt_scene
    .groupby("best_scene")
    .agg(
        對應留言數 = ("comment_id", "count"),
        總讚數     = ("like_count", "sum"),
        最高讚     = ("like_count", "max"),
        平均sim    = ("best_scene_sim", "mean"),
    )
    .reset_index()
    .sort_values("總讚數", ascending=False)
    .rename(columns={"best_scene": "場景"})
)
scene_overview["平均sim"] = scene_overview["平均sim"].round(3)

pd.set_option("display.max_rows", 30)
scene_overview.reset_index(drop=True)

In [ ]:
# ── 修改這兩個變數來選擇場景 ──────────────────────────────────────────
EPISODE      = "ep56"     # 選項："ep56" / "ep63" / "ep76"
SCENE_NUMBER = 894        # 參考上方表格
# ─────────────────────────────────────────────────────────────────────

scene_label = f"{EPISODE}-{SCENE_NUMBER}"
scene_row = scenes[(scenes["episode"] == EPISODE) & (scenes["scene_number"] == SCENE_NUMBER)]

if scene_row.empty:
    print(f"找不到 {scene_label}")
else:
    sr = scene_row.iloc[0]
    matched = cmt_scene[cmt_scene["best_scene"] == scene_label]
    cluster_centroids = analysis[analysis["main_scene"] == scene_label]

    print(f"{'='*65}")
    print(f"{scene_label}")
    print(f"{'='*65}")
    print(f"  對應留言數（個別）: {len(matched)}")
    print(f"  對應 Cluster 數    : {len(cluster_centroids)}")
    print(f"  留言總讚數         : {matched['like_count'].sum():,}")
    print(f"  留言最高讚         : {matched['like_count'].max():,}")
    print(f"\n── 場景原文（前 500 字）──")
    print(sr['text'][:500])

In [ ]:
# 該場景對應的 cluster（centroid 法）
c_table = analysis[analysis["main_scene"] == scene_label][[
    "cluster_id", "size", "likes_total", "likes_max",
    "meme_quality", "main_scene_sim", "keywords"
]].copy()
c_table["meme_quality"]    = c_table["meme_quality"].round(3)
c_table["main_scene_sim"]  = c_table["main_scene_sim"].round(3)
c_table = c_table.sort_values("likes_total", ascending=False).reset_index(drop=True)

print(f"{scene_label} ← {len(c_table)} 個 Cluster 對應到此幕\n")
pd.set_option("display.max_colwidth", 50)
c_table

In [ ]:
# 該場景對應的所有留言（依讚數排序）
TOP_COMMENTS = 30

matched_comments = (
    cmt_scene[cmt_scene["best_scene"] == scene_label]
    .sort_values("like_count", ascending=False)
    [["like_count", "cluster_label", "best_scene_sim", "text_clean"]]
    .rename(columns={"like_count": "讚數", "cluster_label": "cluster",
                     "best_scene_sim": "sim", "text_clean": "留言"})
    .reset_index(drop=True)
)
matched_comments["sim"] = matched_comments["sim"].round(3)

print(f"{scene_label} ← {len(matched_comments)} 則留言，顯示前 {TOP_COMMENTS} 則")
pd.set_option("display.max_colwidth", 100)
matched_comments.head(TOP_COMMENTS)

---
## Part 5：引用 vs 改編分類

每個 cluster 與所有劇本台詞的相似度，分類為：
- **direct_quote**：sim > 0.75，幾乎是原文引用
- **template_modification**：0.5 < sim ≤ 0.75，套格式仿作
- **creative_derivative**：sim ≤ 0.5，脫離原台詞的二創

In [ ]:
# 類型分布
print("▌ Cluster 類型分布\n")
print(quote_cls["type"].value_counts().to_string())

# 各類型按讚數排序
print("\n▌ Direct Quote（接近原文）")
direct = quote_cls[quote_cls["type"] == "direct_quote"].sort_values("likes_total", ascending=False)
print(f"  {len(direct)} 個 cluster")
for _, r in direct.head(10).iterrows():
    print(f"  C{int(r['cluster_id']):>3}  likes={int(r['likes_total']):>6}  sim={r['avg_top_sim']:.3f}")
    print(f"        留言：{str(r['best_comment'])[:80]}")
    print(f"        台詞：{r['matched_character']}：{str(r['matched_line'])[:80]}")

In [ ]:
print("▌ Template Modification（格式仿作）— Top 15\n")
tmpl = quote_cls[quote_cls["type"] == "template_modification"].sort_values("likes_total", ascending=False)
for _, r in tmpl.head(15).iterrows():
    print(f"  C{int(r['cluster_id']):>3}  likes={int(r['likes_total']):>6}  sim={r['avg_top_sim']:.3f}")
    print(f"        留言：{str(r['best_comment'])[:80]}")
    print(f"        台詞：{r['matched_character']}：{str(r['matched_line'])[:60]}")

In [ ]:
print("▌ Creative Derivative（二創）— Top 10\n")
creative = quote_cls[quote_cls["type"] == "creative_derivative"].sort_values("likes_total", ascending=False)
for _, r in creative.head(10).iterrows():
    print(f"  C{int(r['cluster_id']):>3}  likes={int(r['likes_total']):>6}  sim={r['avg_top_sim']:.3f}")
    print(f"        留言：{str(r['best_comment'])[:80]}")

---
## Part 6：場景熱度排行

每個劇情場景被多少 cluster 對應、總讚數、被引用情況。

In [ ]:
print("▌ 場景熱度排行（依總讚數）\n")
pd.set_option("display.max_colwidth", 80)
scene_hm_display = scene_hm[[
    "main_scene", "n_clusters", "total_size", "total_likes",
    "avg_meme_qual", "avg_scene_sim", "top_clusters"
]].copy()
scene_hm_display["avg_meme_qual"] = scene_hm_display["avg_meme_qual"].round(3)
scene_hm_display["avg_scene_sim"] = scene_hm_display["avg_scene_sim"].round(3)
scene_hm_display

In [ ]:
# 顯示 heatmap 圖（如果 PNG 已產出）
from IPython.display import Image, display
png_path = OUTPUT / "scene_heatmap.png"
if png_path.exists():
    display(Image(filename=str(png_path)))
else:
    print(f"{png_path} 不存在，請先跑 scene_heatmap.py")

---
## Part 7：潛力台詞預測

用已知迷因對應的台詞作為正樣本，訓練 LogisticRegression 預測**還沒爆紅但結構上像迷因**的台詞。

In [ ]:
# 訓練資料 vs 預測結果
n_pos = int(potential["is_meme"].sum())
print(f"訓練集：{n_pos} 個正樣本（已知迷因對應台詞）/ {len(potential)} 條台詞")
print(f"\n▌ Top 20 潛力台詞（未在訓練集內，predicted_score 最高）\n")

unlabeled_top = potential[potential["is_meme"] == 0].nlargest(20, "potential_score")
for _, r in unlabeled_top.iterrows():
    print(f"  [{r['potential_score']:.3f}]  {r['character']}：{str(r['text'])[:55]}")
    print(f"           {r['episode']}-{r['scene_number']}  長度={int(r['length'])}")

In [ ]:
# 角色維度分析：哪個角色台詞最容易被預測為高潛力？
print("▌ 各角色平均 potential_score（前 15）\n")
by_char = (
    potential.groupby("character")
    .agg(
        平均潛力分 = ("potential_score", "mean"),
        最高潛力分 = ("potential_score", "max"),
        台詞數     = ("line_id", "count"),
        已知迷因數 = ("is_meme", "sum"),
    )
    .reset_index()
    .sort_values("平均潛力分", ascending=False)
)
by_char["平均潛力分"] = by_char["平均潛力分"].round(3)
by_char["最高潛力分"] = by_char["最高潛力分"].round(3)
by_char.head(15)

In [ ]:
# 在特定場景內看潛力台詞
EPISODE_FILTER     = "ep76"
SCENE_NUMBER_FILTER = 1182

scene_lines = potential[
    (potential["episode"] == EPISODE_FILTER) &
    (potential["scene_number"] == SCENE_NUMBER_FILTER)
].sort_values("potential_score", ascending=False)

print(f"{EPISODE_FILTER}-{SCENE_NUMBER_FILTER} 共 {len(scene_lines)} 行對白\n")
for _, r in scene_lines.head(15).iterrows():
    tag = "★" if r["is_meme"] else " "
    print(f"  {tag} [{r['potential_score']:.3f}]  {r['character']}：{str(r['text'])[:60]}")